# Heatmap Plotting (iRT)
**Summary:** Evaluates iRT predictions from various models for both seen and unseen PTMs.
Calculates 95th percentile absolute differences and visualizes the results in heatmaps.

**Required Files:**
- iRT Predictions parquet files


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib
from matplotlib import pyplot as plt
from glob import iglob
import numpy as np
from typing import List
import re
from matplotlib.colors import BoundaryNorm, ListedColormap
from sklearn import linear_model
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import math
import colorsys
from matplotlib.colors import to_rgb


## Configuration
Paths and variables needed to run this notebook.


In [ ]:
PTMs_PREDICTIONS_GLOB = 'iRT/seen_PTMs/gain_loss/*.parquet'

PLOT_OUTPUT_HEATMAP_SEEN = '<PATH_TO_PLOT_OUTPUT_HEATMAP_SEEN>'
PLOT_OUTPUT_HEATMAP_SEEN_TMT = '<PATH_TO_PLOT_OUTPUT_HEATMAP_SEEN_TMT>'
import os
os.makedirs(os.path.dirname(PLOT_OUTPUT_HEATMAP_SEEN), exist_ok=True)
os.makedirs(os.path.dirname(PLOT_OUTPUT_HEATMAP_SEEN_TMT), exist_ok=True)

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42


## Seen PTMs

In [ ]:
test_ptms_files= [f for f in iglob(PTMs_PREDICTIONS_GLOB)]

for f in test_ptms_files:
    df = pd.read_parquet(f)

    model_name =  f.split('/')[-2]
    package =  f.split('/')[-1].replace('.parquet','')
    df['modified_sequence'] = df['raw_seq'].apply(lambda x: ''.join(x))
    df_fil = df[df['modified_sequence'].str.contains('UNIMOD')]
    print(f)
    print(len(df_fil))

for f in test_ptms_files:
    df = pd.read_parquet(f)
  
    model_name =  f.split('/')[-2]
    package =  f.split('/')[-1].replace('.parquet','')
    if 'modified_sequence' not in df.columns:
        df['modified_sequence'] = df['raw_seq'].apply(lambda x: ''.join(x))
    df_fil = df[df['modified_sequence'].str.contains('UNIMOD')]
    print(f)
    print(len(df_fil))

## iRT Percentile Processing
Calculate the 95th percentile of absolute differences between predictions and actual indexed retention times.

In [ ]:
packages = []
median = []
percentile_95 = []
models = []
for f in test_ptms_files:
    df = pd.read_parquet(f)

    model_name =  f.split('/')[-2]
    package =  f.split('/')[-1].replace('.parquet','')
    if 'modified_sequence' not in df.columns:
        df['modified_sequence'] = df['raw_seq'].apply(lambda x: ''.join(x))
    df_fil = df[df['modified_sequence'].str.contains('UNIMOD')]
   

    df_fil['abs_diff'] = abs(df_fil['irt_pred'] - df_fil['indexed_retention_time'])
    if 'TUM_mod_OGl' in package:
        df_fil_2 = df_fil[df_fil['mod_sequence'].str.contains('S\[')]
        packages.append('TUM_mod_OGL_S')
        median.append(np.percentile(df_fil_2['abs_diff'].values,50))
        percentile_95.append(np.percentile(df_fil_2['abs_diff'].values,95))
        models.append(model_name)

        df_fil_2 = df_fil[df_fil['mod_sequence'].str.contains('T\[')]
        packages.append('TUM_mod_OGL_T')
        median.append(np.percentile(df_fil_2['abs_diff'].values,50))
        percentile_95.append(np.percentile(df_fil_2['abs_diff'].values,95))
        models.append(model_name)

    elif 'TUM_mod_OGa' in package:
        df_fil_2 = df_fil[df_fil['mod_sequence'].str.contains('S\[')]
        packages.append('TUM_mod_OGa_S')
        median.append(np.percentile(df_fil_2['abs_diff'].values,50))
        percentile_95.append(np.percentile(df_fil_2['abs_diff'].values,95))
        models.append(model_name)

        df_fil_2 = df_fil[df_fil['mod_sequence'].str.contains('T\[')]
        packages.append('TUM_mod_OGa_T')
        median.append(np.percentile(df_fil_2['abs_diff'].values,50))
        percentile_95.append(np.percentile(df_fil_2['abs_diff'].values,95))
        models.append(model_name)

    elif 'TUM_mod_mono' in package:
        df_fil_2 = df_fil[df_fil['modified_sequence'].str.contains('K\[')]
        packages.append('TUM_mod_methyl_K')
        median.append(np.percentile(df_fil_2['abs_diff'].values,50))
        percentile_95.append(np.percentile(df_fil_2['abs_diff'].values,80))
        models.append(model_name)

        df_fil_2 = df_fil[df_fil['modified_sequence'].str.contains('R\[')]
        packages.append('TUM_mod_methyl_R')
        median.append(np.percentile(df_fil_2['abs_diff'].values,50))
        percentile_95.append(np.percentile(df_fil_2['abs_diff'].values,80))
        models.append(model_name)

    elif 'pyro' in package:
        df_fil_2 = df_fil[df_fil['mod_sequence'].str.contains('E\[')]
        packages.append('TUM_mod_pyro_E')
        median.append(np.percentile(df_fil_2['abs_diff'].values,50))
        percentile_95.append(np.percentile(df_fil_2['abs_diff'].values,95))
        models.append(model_name)

        df_fil_2 = df_fil[df_fil['mod_sequence'].str.contains('Q\[')]
        packages.append('TUM_mod_pyro_Q')
        median.append(np.percentile(df_fil_2['abs_diff'].values,50))
        percentile_95.append(np.percentile(df_fil_2['abs_diff'].values,95))
        models.append(model_name)

    else:
        packages.append(package)
        median.append(np.percentile(df_fil['abs_diff'].values,50))
        percentile_95.append(np.percentile(df_fil['abs_diff'].values,80))
        models.append(model_name)

df_irt_pred = pd.DataFrame()
df_irt_pred['package'] = packages
df_irt_pred['median'] = median
df_irt_pred['model'] = models
df_irt_pred['percentile_95'] = percentile_95

test_ptms_files= [f for f in iglob(SEEN_PTM_GLOB) if '_pred' in f and 'naive' in f] #and 'mod' in f and 'TMT' not in f and 'imp' not in f]

df = pd.read_parquet(test_ptms_files[0])

df['package'] = df['package'].str.replace(r'(_\d+)+$', '', regex=True)
df['package'] = df['package'].str.replace('TUM_mod_citrullination_l', 'TUM_mod_citrullination')
df['package'] = df['package'].str.replace('TUM_mod_citrullination_s', 'TUM_mod_citrullination')

In [ ]:
df_groups = df.groupby('package')
for package,g in df_groups:
    path = '/'.join(test_ptms_files[0].split('/')[:-1]) + '/' + package + '.parquet'
    if 'mod' in package or 'nterm' in package:
        g.to_parquet(path,index=False)

In [ ]:
packages = []
median = []
percentile_95 = []
models = []
for f in test_ptms_files:
    df = pd.read_parquet(f)
    df['package'] = df['package'].str.replace(r'(_\d+)+$', '', regex=True)
    df['package'] = df['package'].str.replace('TUM_mod_citrullination_l', 'TUM_mod_citrullination')
    df['package'] = df['package'].str.replace('TUM_mod_citrullination_s', 'TUM_mod_citrullination')
    for package in df['package'].unique():
        
        model_name =  f.split('/')[-2]
        if 'old' in model_name:
            continue
        df_fil = df[df['package'] == package]
        regr = linear_model.LinearRegression()
        df_fil['abs_diff'] = abs(df_fil['pred'] - df_fil['indexed_retention_time'])
        df_fil['mod_sequence'] = df_fil['raw_seq'].apply(lambda x: ''.join(x))

        #if model_name =='naive':
        if 'TUM_mod_OGl' in package:
            df_fil_2 = df_fil[df_fil['mod_sequence'].str.contains('S\[')]
            packages.append('OGL_S')
            median.append(np.percentile(df_fil_2['abs_diff'].values,50))
            percentile_95.append(np.percentile(df_fil_2['abs_diff'].values,95))
            models.append(model_name)

            df_fil_2 = df_fil[df_fil['mod_sequence'].str.contains('T\[')]
            packages.append('OGL_T')
            median.append(np.percentile(df_fil_2['abs_diff'].values,50))
            percentile_95.append(np.percentile(df_fil_2['abs_diff'].values,95))
            models.append(model_name)

        elif 'TUM_mod_OGa' in package:
            df_fil_2 = df_fil[df_fil['mod_sequence'].str.contains('S\[')]
            packages.append('OGa_S')
            median.append(np.percentile(df_fil_2['abs_diff'].values,50))
            percentile_95.append(np.percentile(df_fil_2['abs_diff'].values,95))
            models.append(model_name)

            df_fil_2 = df_fil[df_fil['mod_sequence'].str.contains('T\[')]
            packages.append('OGa_T')
            median.append(np.percentile(df_fil_2['abs_diff'].values,50))
            percentile_95.append(np.percentile(df_fil_2['abs_diff'].values,95))
            models.append(model_name)

        elif 'TUM_mod_mono' in package:
            df_fil_2 = df_fil[df_fil['mod_sequence'].str.contains('K\[')]
            packages.append('methyl_K')
            median.append(np.percentile(df_fil_2['abs_diff'].values,50))
            percentile_95.append(np.percentile(df_fil_2['abs_diff'].values,95))
            models.append(model_name)

            df_fil_2 = df_fil[df_fil['mod_sequence'].str.contains('R\[')]
            packages.append('methyl_R')
            median.append(np.percentile(df_fil_2['abs_diff'].values,50))
            percentile_95.append(np.percentile(df_fil_2['abs_diff'].values,95))
            models.append(model_name)

        elif 'pyro' in package:
            df_fil_2 = df_fil[df_fil['mod_sequence'].str.contains('E\[')]
            packages.append('pyro_E')
            median.append(np.percentile(df_fil_2['abs_diff'].values,50))
            percentile_95.append(np.percentile(df_fil_2['abs_diff'].values,95))
            models.append(model_name)

            df_fil_2 = df_fil[df_fil['mod_sequence'].str.contains('Q\[')]
            packages.append('pyro_Q')
            median.append(np.percentile(df_fil_2['abs_diff'].values,50))
            percentile_95.append(np.percentile(df_fil_2['abs_diff'].values,95))
            models.append(model_name)

        else:
            packages.append(package)
            median.append(np.percentile(df_fil['abs_diff'].values,50))
            percentile_95.append(np.percentile(df_fil['abs_diff'].values,95))
            models.append(model_name)

df_irt_pred = pd.DataFrame()
df_irt_pred['package'] = packages
df_irt_pred['median'] = median
df_irt_pred['model'] = models
df_irt_pred['percentile_95'] = percentile_95

df_irt_pred[df_irt_pred['package'].str.contains('nterm')]

## Heatmap Visualization
Pivot the percentile values into a matrix and plot them as a heatmap comparing models across modifications.

In [ ]:
pt = pd.pivot_table(df_irt_pred, 
                    index='model', values=['percentile_95'], columns='package')

pt.columns = pt.columns.get_level_values(1)
pt.index = pd.CategoricalIndex(pt.index, categories= ['basic','naive', 'delta_mass','ac', 'gain_loss'])
pt.sort_index(level=0, inplace=True)



fig, ax = plt.subplots(figsize = (16, 6))
ax.set_title("Seen Mods Spectral Angle")
bounds = np.concatenate([
    np.linspace(2, 4, 50),
    np.linspace(4, 7, 35)[1:],
    np.linspace(7, 10, 30)[1:],      # e.g. [0.0, 0.233, 0.467, 0.7]
    np.linspace(10, 50, 60)[1:],
    #np.linspace(0.9, 0.93, 50)[1:]     # e.g. [0.743, 0.786, 0.829, 0.871, 0.914, 0.957, 1.0]
])
# Total number of colors needed = len(bounds) - 1
n_bins = len(bounds) - 1

# Discretize the colormap to n_bins colors
base_cmap = plt.get_cmap('mako_r')
discrete_cmap = ListedColormap(base_cmap(np.linspace(0, 1, n_bins)))

# Create the norm
norm = BoundaryNorm(boundaries=bounds, ncolors=300)
ticks_to_show = [2, 4, 7, 10, 40]
sns.heatmap(pt, cmap="mako_r", annot=True, norm=norm, cbar_kws={'ticks': ticks_to_show, 'format': '%.2f'}, xticklabels=1, yticklabels=1,vmin=0.12,vmax=0.93)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(PLOT_OUTPUT_HEATMAP_SEEN_TMT)